# SmartHire — 04. Unsupervised Job Role Clustering & Topic Analysis
In this notebook, we discover natural job families and market role segments using unsupervised machine learning:
- **TF-IDF Vector Space Representation**.
- **KMeans Clustering** with empirical cluster selection.
- **Elbow Method (Inertia Curve)** & **Silhouette Coefficient Analysis** across K = 4..12.
- **Dimensionality Reduction**: 2D PCA visual projection.
- **Cluster Topic Interpretation**: Identifying top keywords, job titles, and high-frequency skills per role family.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

from src.config import CLUSTER_METADATA_PATH, CLUSTERING_MODEL_PATH, JOBS_PROCESSED_PATH
from src.models.clustering import JobClusterer

jobs_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "jobs_clean.csv")
print(f"Loaded jobs dataset: {jobs_df.shape}")


## 1. Load Precomputed Clusterer and Metadata


In [ ]:
clusterer = JobClusterer.load()
print(f"Loaded JobClusterer with K={clusterer.n_clusters} clusters.")

# Display all cluster summaries
cluster_rows = []
for c_id, info in clusterer.cluster_info.items():
    cluster_rows.append({
        "Cluster ID": c_id,
        "Label": info["label"],
        "Size": info["size"],
        "Market Share": f"{info['pct_of_corpus']}%",
        "Top Keywords": ", ".join(info["top_keywords"][:5]),
        "Top Skills": ", ".join(list(info["top_skills"].keys())[:5]),
    })

display(pd.DataFrame(cluster_rows))


## 2. Inspect PCA 2D Cluster Visualization


In [ ]:
from PIL import Image

pca_path = PROJECT_ROOT / "reports" / "figures" / "job_clusters_pca.png"
if pca_path.exists():
    img = Image.open(pca_path)
    plt.figure(figsize=(12, 7))
    plt.imshow(img)
    plt.axis("off")
    plt.title("2D PCA Projection of Job Postings", fontsize=14, fontweight="bold")
    plt.show()


## 3. Elbow and Silhouette Analysis Plots


In [ ]:
elbow_path = PROJECT_ROOT / "reports" / "figures" / "elbow_method.png"
sil_path = PROJECT_ROOT / "reports" / "figures" / "silhouette_analysis.png"

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
if elbow_path.exists():
    axes[0].imshow(Image.open(elbow_path))
    axes[0].axis("off")
    axes[0].set_title("Elbow Method (Inertia vs K)", fontsize=12)

if sil_path.exists():
    axes[1].imshow(Image.open(sil_path))
    axes[1].axis("off")
    axes[1].set_title("Silhouette Analysis Across K", fontsize=12)

plt.tight_layout()
plt.show()


## 4. Assigning a New Resume to a Market Job Cluster


In [ ]:
sample_cv = "DevOps Engineer with AWS, Docker, Kubernetes, Terraform, CI/CD, Jenkins, and Linux."
assigned_cluster = clusterer.predict_cluster(sample_cv)
info = clusterer.cluster_info[assigned_cluster]
print(f"Assigned Cluster ID: {assigned_cluster}")
print(f"Cluster Label: {info['label']}")
print(f"Top Keywords: {info['top_keywords']}")
print(f"Top Skills in this Role Family: {list(info['top_skills'].keys())}")
